In [22]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.linear_model import LinearRegression
from statsmodels.tools.eval_measures import rmse

In [15]:
# Load the dataset into a pandas DataFrame
data = pd.read_csv('../data/housing_data.csv')
data.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [16]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             545 non-null    int64 
 1   area              545 non-null    int64 
 2   bedrooms          545 non-null    int64 
 3   bathrooms         545 non-null    int64 
 4   stories           545 non-null    int64 
 5   mainroad          545 non-null    object
 6   guestroom         545 non-null    object
 7   basement          545 non-null    object
 8   hotwaterheating   545 non-null    object
 9   airconditioning   545 non-null    object
 10  parking           545 non-null    int64 
 11  prefarea          545 non-null    object
 12  furnishingstatus  545 non-null    object
dtypes: int64(6), object(7)
memory usage: 55.5+ KB


In [17]:
# Separate the features (X) and target variable (y)
X = data.drop('price', axis=1)
y = data['price']

# Perform any necessary preprocessing steps
# For example, handle missing values, encode categorical variables, scale numerical features, etc.
# Here, we'll assume that X contains a mix of numerical and categorical features
numerical_features = ['area','bedrooms', 'bathrooms','stories','parking']
categorical_features = ['mainroad','guestroom','basement','hotwaterheating','airconditioning','prefarea','furnishingstatus']

In [6]:
X

,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...
540,3000,2,1,1,yes,no,yes,no,no,2,no,unfurnished
541,2400,3,1,1,no,no,no,no,no,0,no,semi-furnished
542,3620,2,1,1,yes,no,no,no,no,0,no,unfurnished
543,2910,3,1,1,no,no,no,no,no,0,no,furnished


In [7]:
y

0      13300000
1      12250000
2      12250000
3      12215000
4      11410000
         ...   
540     1820000
541     1767150
542     1750000
543     1750000
544     1750000
Name: price, Length: 545, dtype: int64

In [19]:
# Define the preprocessing steps for numerical and categorical features
numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder()

# Combine the preprocessing steps using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Define the regression model
model = LinearRegression()

# Create the pipeline by combining the preprocessing and modeling steps
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

In [20]:
# Choose the appropriate cross-validation technique (e.g., k-fold cross-validation)
# Here, we'll use 5-fold cross-validation
cv = 5

# Use cross_val_score to perform cross-validation and obtain the evaluation scores
scores = cross_val_score(pipeline, X, y, scoring='neg_mean_squared_error', cv=cv)

# Convert the negative mean squared error scores to positive values
mse_scores = -scores

# Compute the mean and standard deviation of the cross-validation scores
mean_mse = np.mean(mse_scores)
std_mse = np.std(mse_scores)

# Print the mean and standard deviation of the cross-validation scores
print(f"Mean MSE: {mean_mse}")
print(f"Standard Deviation of MSE: {std_mse}")

Mean MSE: 2120447069698.9917
Standard Deviation of MSE: 2370083656888.461


<h2>cross_val_score does not remember trained parameters for each folds</h2>

<h2>Train final model and  Final evaluation</h2>

In [25]:
# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit the pipeline to the training data
pipeline.fit(X_train, y_train)

# Evaluate the performance of the model on the validation set
val_predictions = pipeline.predict(X_val)
rmse_val = rmse(y_val, val_predictions)
r2=pipeline.score(X_val, y_val)


# Print the evaluation metrics
print(f"MSE on Validation Set: {rmse_val}")
print(f"R-squared on Validation Set: {r2}")

MSE on Validation Set: 1324506.9600914407
R-squared on Validation Set: 0.6529242642153172


⚠️ What you are doing now

You are:
	1.	Doing cross-validation on ALL data
	2.	Then splitting and evaluating again

👉 This is not wrong, but:
	•	your validation set is no longer truly “unseen”
	•	it’s slightly optimistic


By following these steps, you can implement the pipeline with cross-validation using scikit-learn. The pipeline incorporates the necessary preprocessing steps, the regression model, and performs cross-validation to assess the model’s performance. It also includes a separate evaluation step on a validation set to provide additional insights into the model’s performance on unseen data.

Interpretation and Analysis of the Results
After implementing the pipeline with cross-validation and training the regression model, it’s important to interpret and analyze the results to gain insights into the model’s performance and understand the patterns in the data. Here are some steps to guide the interpretation and analysis process:

Cross-Validation Results: Review the mean squared error (MSE) obtained from cross-validation. A lower MSE indicates better predictive performance. Assess the consistency of the performance across the folds by considering the standard deviation of the MSE scores. A lower standard deviation suggests a more stable model.

Coefficient Analysis: If you are using a linear regression model, interpret the coefficients to identify the impact of each feature on the predicted housing prices. Positive coefficients indicate a positive relationship, while negative coefficients suggest a negative relationship. The magnitude of the coefficients represents the strength of the influence.

Residual Analysis: Examine the residuals (the differences between the predicted and actual sale prices) to identify any patterns or biases in the model’s predictions. Plot the residuals against the predicted values to check for heteroscedasticity (unequal variance) or any systematic errors in the predictions. Addressing these patterns may lead to improvements in the model.

Validation Set Metrics: Evaluate the model’s performance on the validation set. Compare the mean squared error (MSE) and R-squared values to assess how well the model generalizes to unseen data. A lower MSE and a higher R-squared indicate better performance.

By interpreting and analyzing the results, you can gain valuable insights into the model’s performance and the underlying patterns in the data. These insights can guide further improvements in the pipeline, feature selection, or model selection to enhance the predictive power of the regression model.